# 4.2 Messages In, a Message Out: Concepts Notebook

**Generative AI Foundations · Module 4**

| # | Concept |
|---|---|
| 1 | The one rule of chat models |
| 2 | Three ways to send input |
| 3 | Message classes and roles |
| 4 | What comes back: the `AIMessage` |
| 5 | `.text` vs `.content` vs `.content_blocks` |
| 6 | Provider differences |
| 7 | Model profiles |
| 8 | Many questions at once: `batch` |

**Before you start:** Ollama is running and `gemma3:1b` is pulled.

## 0. Setup

In [ ]:
%pip install -q "langchain[ollama,google-genai]" python-dotenv

In [ ]:
import os

from langchain.chat_models import init_chat_model

llm = init_chat_model("ollama:gemma3:1b", temperature=0.7)
print("Ready:", type(llm).__name__)

---
## 1. The One Rule of Chat Models

Every chat model, from every provider, works the same way:

```text
messages in  --->  [ chat model ]  --->  one AIMessage out
```

> **Mental Model: The Post Office**
> You post a bundle of letters (messages). You get **one** reply letter back, inside an envelope with stamps on it.

In [ ]:
reply = llm.invoke("In one sentence, what is an API?")

print(type(reply).__name__)
print(reply.text)

---
## 2. Three Ways to Send Input

All three become a **list of messages** before reaching the model.

```text
"What is Python?"                          --->  [ user: "What is Python?" ]
[{"role": "user", "content": "..."}]       --->  [ user: "..." ]
[HumanMessage("...")]                      --->  [ user: "..." ]
```

Let's see the conversion **without calling a model**:

In [ ]:
from langchain.messages import HumanMessage, SystemMessage
from langchain_core.messages.utils import convert_to_messages

inputs = {
    "string": "What is Python?",
    "dicts": [{"role": "system", "content": "Be brief."}, {"role": "user", "content": "What is Python?"}],
    "objects": [SystemMessage("Be brief."), HumanMessage("What is Python?")],
}

for name, value in inputs.items():
    value = [value] if isinstance(value, str) else value
    print(f"{name:<8} ->", [(m.type, m.text) for m in convert_to_messages(value)])

The dictionaries and the objects produce exactly the same messages. Now with the model:

In [ ]:
print(llm.invoke("What is Python? One sentence.").text)
print(llm.invoke([{"role": "system", "content": "Be brief."}, {"role": "user", "content": "What is Python?"}]).text)
print(llm.invoke([SystemMessage("Be brief."), HumanMessage("What is Python?")]).text)

---
## 3. Message Classes and Roles

| Class | Role | Who writes it |
|---|---|---|
| `SystemMessage` | system | You, the developer |
| `HumanMessage` | user | The person using the app |
| `AIMessage` | assistant | The model |
| `ToolMessage` | tool | Your Python functions (Lesson 4.5) |

> **Mental Model: Name Badges**
> Each message wears a badge saying who wrote it. The model reads the badges to understand the conversation.

In [ ]:
from langchain.messages import AIMessage

conversation = [
    SystemMessage("You are a friendly tutor."),
    HumanMessage("Hi!"),
    AIMessage("Hello! What would you like to learn?"),
]

for message in conversation:
    print(f"{type(message).__name__:<14} badge = {message.type:<6} text = {message.text}")

---
## 4. What Comes Back: the `AIMessage`

> **Mental Model: The Letter and the Envelope**
> `.text` is the letter. `.usage_metadata` is the postage (tokens). `.response_metadata` is the stamps (model, provider, why it stopped).

```text
AIMessage
├── .text               "Python is ..."
├── .usage_metadata     {input_tokens, output_tokens, total_tokens}
├── .response_metadata  {model_name, model_provider, done_reason, ...}
└── .id                 "lc_run--..."
```

In [ ]:
reply = llm.invoke("Name two uses of Python, comma separated.")

print("Text:    ", reply.text)
print("Tokens:  ", reply.usage_metadata)
print("Model:   ", reply.response_metadata.get("model_name"))
print("Provider:", reply.response_metadata.get("model_provider"))
print("ID:      ", reply.id)

**Example 2:** tokens grow when the question grows.

In [ ]:
short = llm.invoke("Hi")
long = llm.invoke("Please explain, in one sentence, why Python is popular with beginners and with data scientists.")

print("Short question input tokens:", short.usage_metadata["input_tokens"])
print("Long question input tokens: ", long.usage_metadata["input_tokens"])

---
## 5. `.text` vs `.content` vs `.content_blocks`

Some models "think" before answering. Their `.content` is then a **list**, not a string.

| Attribute | Gives you |
|---|---|
| `.content` | Raw data: a string **or** a list (depends on the provider) |
| `.text` | Always just the answer, as a string |
| `.content_blocks` | Every part, labelled in **one standard format** |

> **Mental Model: The Mixed Envelope**
> Some envelopes contain the letter plus the writer's rough notes. `.text` hands you only the letter. `.content_blocks` lays out everything, each piece labelled.

No model is needed for this example: we build such a reply ourselves.

In [ ]:
reply = AIMessage(
    content=[
        {"type": "thinking", "thinking": "Keep it short.", "signature": "abc"},
        {"type": "text", "text": "Python is a popular programming language."},
    ],
    response_metadata={"model_provider": "anthropic"},
)

print(".content is a", type(reply.content).__name__)
print(".text        :", reply.text)
print(".content_blocks:")
for block in reply.content_blocks:
    print("   ", block["type"], "->", block.get("text") or block.get("reasoning"))

The provider called it `thinking`; LangChain's standard name is `reasoning`. **Rule: use `.text` for the answer.**

---
## 6. Provider Differences

The same idea can have a different name per provider. Example: **why the model stopped writing**.

| Provider | Key |
|---|---|
| Ollama | `done_reason` |
| OpenAI, Gemini | `finish_reason` |
| Anthropic | `stop_reason` |

A small helper handles all of them:

In [ ]:
def finish_reason(message) -> str:
    metadata = message.response_metadata
    return metadata.get("finish_reason") or metadata.get("done_reason") or metadata.get("stop_reason") or "unknown"


reply = llm.invoke("Say OK.")
print("Finish reason:", finish_reason(reply))

---
## 7. Model Profiles

A **profile** lists what a model can do: context size, image input, tool calling, and more.

> **Mental Model: The Spec Sheet on the Box**

Reading a profile does **not** call the model, so it is free.

In [ ]:
print("Ollama gemma3:1b profile:", llm.profile)

Local models often have no profile. A cloud model usually does (any placeholder key works here, because no request is sent):

In [ ]:
gemini = init_chat_model("google_genai:gemini-3.5-flash-lite", api_key=os.getenv("GOOGLE_API_KEY", "placeholder-key"))

profile = gemini.profile or {}
for key in ["max_input_tokens", "max_output_tokens", "image_inputs", "tool_calling"]:
    print(f"{key:<18} {profile.get(key)}")

Always write `llm.profile or {}`, because the profile can be missing.

---
## 8. Many Questions at Once: `batch`

> **Mental Model: One Waiter, Many Orders**
> `invoke` is a waiter taking one order to the kitchen and waiting. `batch` takes several orders at once; the kitchen works on them together, and the dishes come back **in the same order**.

```text
invoke:  Q1 -> A1, then Q2 -> A2, then Q3 -> A3
batch:   [Q1, Q2, Q3] -> [A1, A2, A3]
```

In [ ]:
questions = ["What is Python? Five words.", "What is an API? Five words.", "What is an LLM? Five words."]

replies = llm.batch(questions, config={"max_concurrency": 3})

for question, reply in zip(questions, replies):
    print(f"{question:<30} -> {reply.text}")

---
## Summary

| Concept | Remember |
|---|---|
| The rule | Messages in, one `AIMessage` out |
| Input forms | String, list of dicts, or list of message objects |
| Roles | system, user (human), assistant (ai), tool |
| `.text` | The answer as a string: use it by default |
| `.content_blocks` | Every part in one standard format |
| Metadata | Tokens in `.usage_metadata`; provider details in `.response_metadata` (names differ) |
| `.profile` | The model's spec sheet; may be missing |
| `batch` | Many requests at once; results in the same order |

## Practice

1. In Section 4, ask three questions of different lengths and print only the token counts.
2. In Section 3, add a fourth message: `HumanMessage("What did you just ask me?")`, then send the whole list to `llm.invoke`. What does it answer?
3. In Section 8, add two more questions and compare the time of `batch` with a normal `for` loop using `invoke`.